# 🔍 TP : Détection d'Outliers avec la Méthode IQR
## Utilisation de notre module `ifri_mini_ml_lib`

**Groupe 4** — Dataset : Credit Card Fraud Detection

---

### 🎯 Objectif
Ce notebook montre comment utiliser **notre propre implémentation** de la méthode **IQR (Interquartile Range)** pour détecter des valeurs aberrantes (outliers) sur la variable `Amount` du dataset `creditcard.csv`.

### 📋 Plan du notebook
1. **Importation** du module IQR (avec fallback via chemin local)
2. **Chargement et exploration** du dataset
3. **Détection des outliers** avec notre IQR
4. **Visualisation** des résultats
5. **Évaluation** des performances (comparaison avec les vraies fraudes)

## Étape 1 : Importation des bibliothèques

On importe d'abord les bibliothèques classiques (numpy, pandas, matplotlib, seaborn).

Puis on tente d'importer notre classe `IQR` depuis le package `ifri_mini_ml_lib`.

> **Si le package n'est pas encore installé** (car pas encore publié sur PyPI), le `try/except` récupère l'erreur et ajoute le chemin local du projet au `sys.path` pour que l'import fonctionne quand même.

In [ ]:
# ============================================================
# IMPORTS GÉNÉRAUX
# ============================================================
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')  # On désactive les warnings mineurs pour un affichage propre

# ============================================================
# PRIORISATION DU CHEMIN LOCAL & NETTOYAGE DU CACHE
# ============================================================
# Pour éviter que Python n'utilise une version installée via pip (site-packages)
# ou qu'il utilise des modules déjà mis en cache lors d'une précédente exécution :

# 1. On nettoie tout cache lié à 'ifri_mini_ml_lib'
for key in list(sys.modules.keys()):
    if key.startswith("ifri_mini_ml_lib"):
        del sys.modules[key]

# 2. On s'assure que notre chemin local est TOUJOURS au tout début de sys.path (index 0)
# Même si la cellule est réexécutée plusieurs fois.
CHEMIN_PROJET = r"D:\DEV-Project\ifri_mini_ml_lib"
while CHEMIN_PROJET in sys.path:
    sys.path.remove(CHEMIN_PROJET)
sys.path.insert(0, CHEMIN_PROJET)

# ============================================================
# IMPORT DE NOTRE CLASSE IQR
# ============================================================
try:
    from ifri_mini_ml_lib.anomalies_detection.iqr import IQR
    import ifri_mini_ml_lib
    print("✅ Module IQR chargé avec succès !")
    print(f"📍 Emplacement physique du module : {os.path.abspath(ifri_mini_ml_lib.__file__)}")
except ImportError as e:
    print(f"❌ Erreur lors de l'importation : {e}")
    print(f"Veuillez vérifier l'existence du dossier racine : {CHEMIN_PROJET}")

## Étape 2 : Chargement et exploration du dataset

Le dataset **Credit Card Fraud Detection** contient ~285 000 transactions bancaires.

- La colonne `Amount` représente le montant de chaque transaction.
- La colonne `Class` contient les étiquettes : **0 = normal**, **1 = fraude**.

Nous allons travailler **uniquement sur la variable `Amount`** pour détecter les outliers avec notre IQR.

In [ ]:
# ============================================================
# CHARGEMENT DU DATASET
# ============================================================
# Le fichier creditcard.csv doit être dans le même dossier que ce notebook.
df = pd.read_csv("creditcard.csv")

# Affichage des informations de base du dataset
print("="*60)
print("   INFORMATIONS SUR LE DATASET")
print("="*60)
print(f"Nombre de transactions   : {len(df):,}")
print(f"Nombre de colonnes       : {df.shape[1]}")
print(f"Transactions normales    : {(df['Class'] == 0).sum():,}")
print(f"Transactions frauduleuses: {(df['Class'] == 1).sum():,}")
print(f"Taux de fraude           : {df['Class'].mean()*100:.3f}%")
print("="*60)

# Aperçu des premières lignes
df.head()

In [ ]:
# ============================================================
# EXPLORATION DE LA VARIABLE "Amount"
# ============================================================
# On affiche les statistiques descriptives de la colonne Amount
# pour comprendre sa distribution avant la détection.
print("Statistiques descriptives de la variable 'Amount' :\n")
print(df['Amount'].describe().to_string())

print(f"\n📊 Médiane (Q2)  : {df['Amount'].median():.2f} €")
print(f"📊 Variance     : {df['Amount'].var():.2f}")
print(f"📊 Asymétrie     : {df['Amount'].skew():.2f}  (>0 = queue à droite)")

In [ ]:
# ============================================================
# VISUALISATION DE LA DISTRIBUTION DE Amount (AVANT DÉTECTION)
# ============================================================
# On trace un histogramme + un boxplot côte à côte pour voir la distribution.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme (on limite à 500€ pour mieux voir la distribution principale)
axes[0].hist(df['Amount'], bins=100, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution de Amount', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Montant (€)')
axes[0].set_ylabel('Fréquence')
axes[0].set_xlim(0, 500)  # Zoom sur les valeurs < 500€
axes[0].axvline(df['Amount'].median(), color='#e74c3c', linestyle='--', label=f"Médiane = {df['Amount'].median():.1f}€")
axes[0].legend()

# Boxplot
axes[1].boxplot(df['Amount'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#3498db', alpha=0.6),
                medianprops=dict(color='#e74c3c', linewidth=2))
axes[1].set_title('Boxplot de Amount', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Montant (€)')

plt.suptitle('Exploration visuelle de la variable Amount', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("ℹ️  Le boxplot montre déjà que beaucoup de valeurs extrêmes sont présentes.")

## Étape 3 : Détection des outliers avec notre IQR

### Rappel théorique de la méthode IQR

L'**IQR (Interquartile Range)** est défini comme la différence entre le 3ème quartile (Q3) et le 1er quartile (Q1) :

$$IQR = Q3 - Q1$$

Une observation est considérée comme un **outlier** si elle tombe en dehors de l'intervalle :

$$[Q1 - k \times IQR, \; Q3 + k \times IQR]$$

où `k` est le facteur multiplicatif (par défaut **1.5**).

### Utilisation de notre module

Notre classe `IQR` suit l'API scikit-learn :
1. `detector = IQR(factor=1.5)` → création du détecteur
2. `detector.fit(X)` → apprentissage des bornes sur les données
3. `detector.predict(X)` → retourne **0** (normal) ou **1** (outlier) pour chaque observation

In [ ]:
# ============================================================
# PRÉPARATION DES DONNÉES
# ============================================================
# On extrait uniquement la colonne "Amount" sous forme de tableau numpy 2D.
# Notre module IQR attend un tableau de forme (n_samples, n_features).
# Ici on n'a qu'une seule feature (Amount), donc la forme sera (284807, 1).

X = df[['Amount']].values  # Shape: (n, 1) — tableau 2D obligatoire pour notre IQR

# On sauvegarde aussi les vraies étiquettes pour l'évaluation plus tard
y_true = df['Class'].values  # 0 = normal, 1 = fraude

print(f"Forme des données X : {X.shape}")
print(f"Forme des étiquettes : {y_true.shape}")
print(f"Exemple de valeurs X : {X[:5].flatten()}")

In [ ]:
# ============================================================
# CRÉATION ET ENTRAÎNEMENT DU DÉTECTEUR IQR
# ============================================================
# On crée une instance de notre détecteur IQR avec le facteur classique 1.5.
# Puis on appelle fit() pour qu'il calcule Q1, Q3, IQR et les bornes.

detector = IQR(factor=1.5)

# fit() calcule les statistiques (Q1, Q3, IQR, bornes inférieure et supérieure)
detector.fit(X)

# Affichage des bornes apprises par le modèle
print("="*55)
print("   BORNES APPRISES PAR NOTRE DÉTECTEUR IQR")
print("="*55)
print(f"   Premier quartile (Q1)      : {detector.Q1_[0]:.2f} €")
print(f"   Troisième quartile (Q3)     : {detector.Q3_[0]:.2f} €")
print(f"   Écart Interquartile (IQR)    : {detector.IQR_[0]:.2f} €")
print(f"   Facteur multiplicatif (k)   : {detector.factor}")
print("   " + "-"*50)

# Récupération des bornes via la méthode get_bounds()
bounds = detector.get_bounds()
borne_inf = bounds['lower'][0]
borne_sup = bounds['upper'][0]

print(f"   🟢 Borne inférieure : {borne_inf:.2f} €")
print(f"   🟢 Borne supérieure : {borne_sup:.2f} €")
print(f"\n   → Toute transaction en dehors de [{borne_inf:.2f}, {borne_sup:.2f}]")
print(f"     sera considérée comme un OUTLIER.")
print("="*55)

In [ ]:
# ============================================================
# PRÉDICTION : DÉTECTION DES OUTLIERS
# ============================================================
# La méthode predict() retourne un tableau de 0 et 1 :
#   - 0 = transaction normale (dans l'intervalle [borne_inf, borne_sup])
#   - 1 = outlier (en dehors de l'intervalle)

y_pred = detector.predict(X)

# Ajout des prédictions dans le DataFrame pour faciliter l'analyse
df['IQR_Outlier'] = y_pred

# Résumé des résultats
n_outliers = y_pred.sum()
n_normaux = len(y_pred) - n_outliers

print("="*55)
print("   RÉSULTATS DE LA DÉTECTION")
print("="*55)
print(f"   Transactions normales : {n_normaux:,}  ({n_normaux/len(y_pred)*100:.2f}%)")
print(f"   Outliers détectés     : {n_outliers:,}  ({n_outliers/len(y_pred)*100:.2f}%)")
print("="*55)

# Aperçu de quelques outliers détectés
print("\n🔴 Exemples de transactions identifiées comme outliers :")
print(df[df['IQR_Outlier'] == 1][['Amount', 'Class', 'IQR_Outlier']].head(10).to_string(index=True))

## Étape 4 : Visualisation des résultats

On va tracer plusieurs graphiques pour comprendre visuellement ce que notre détecteur IQR a identifié :

1. **Scatter plot** des transactions colorées par label (normal vs outlier)
2. **Boxplot** avec les bornes IQR matérialisées
3. **Distribution** comparée des montants normaux vs outliers

In [ ]:
# ============================================================
# GRAPHIQUE 1 : SCATTER PLOT DES TRANSACTIONS
# ============================================================
# Chaque point représente une transaction.
# Axe X = index de la transaction, Axe Y = montant.
# Les outliers sont colorés en rouge, les normaux en bleu.

fig, ax = plt.subplots(figsize=(14, 6))

# Points normaux (en bleu, petits et semi-transparents car très nombreux)
normaux = df[df['IQR_Outlier'] == 0]
outliers = df[df['IQR_Outlier'] == 1]

ax.scatter(normaux.index, normaux['Amount'], c='#3498db', s=3, alpha=0.3, label=f'Normal ({len(normaux):,})')
ax.scatter(outliers.index, outliers['Amount'], c='#e74c3c', s=20, alpha=0.8, label=f'Outlier ({len(outliers):,})', marker='x')

# Lignes des bornes IQR
ax.axhline(y=borne_sup, color='#e67e22', linestyle='--', linewidth=2, label=f'Borne supérieure = {borne_sup:.2f}€')
ax.axhline(y=borne_inf, color='#27ae60', linestyle='--', linewidth=2, label=f'Borne inférieure = {borne_inf:.2f}€')

ax.set_title('Détection des outliers sur la variable Amount (méthode IQR)', fontsize=14, fontweight='bold')
ax.set_xlabel('Index de la transaction', fontsize=12)
ax.set_ylabel('Montant (€)', fontsize=12)
ax.legend(loc='upper right', fontsize=10, frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRAPHIQUE 2 : BOXPLOT ANNOTÉ AVEC LES BORNES IQR
# ============================================================
# Ce graphique montre le boxplot classique de Amount.
# On y superpose nos bornes IQR pour bien voir où elles tombent.

fig, ax = plt.subplots(figsize=(8, 6))

bp = ax.boxplot(df['Amount'], vert=True, patch_artist=True, widths=0.5,
                boxprops=dict(facecolor='#3498db', alpha=0.5),
                medianprops=dict(color='#e74c3c', linewidth=2.5),
                whiskerprops=dict(color='#2c3e50', linewidth=1.5),
                capprops=dict(color='#2c3e50', linewidth=1.5),
                flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#e74c3c'))

# Annotation des bornes de notre IQR
ax.axhline(y=borne_sup, color='#e67e22', linestyle='--', linewidth=2)
ax.text(1.3, borne_sup, f'Borne sup = {borne_sup:.1f}€', fontsize=10, color='#e67e22', fontweight='bold')

ax.axhline(y=borne_inf, color='#27ae60', linestyle='--', linewidth=2)
ax.text(1.3, borne_inf, f'Borne inf = {borne_inf:.1f}€', fontsize=10, color='#27ae60', fontweight='bold')

ax.set_title('Boxplot de Amount avec bornes IQR', fontsize=14, fontweight='bold')
ax.set_ylabel('Montant (€)', fontsize=12)
ax.set_xticklabels(['Amount'])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# GRAPHIQUE 3 : HISTOGRAMME COMPARÉ (NORMAUX vs OUTLIERS)
# ============================================================
# On compare la distribution des montants des transactions normales
# vs celles identifiées comme outliers.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des normaux (zoom sur 0-200€)
axes[0].hist(normaux['Amount'], bins=80, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Transactions normales (n={len(normaux):,})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Montant (€)')
axes[0].set_ylabel('Fréquence')
axes[0].set_xlim(0, 200)
axes[0].axvline(normaux['Amount'].median(), color='#e74c3c', linestyle='--',
                label=f"Médiane = {normaux['Amount'].median():.1f}€")
axes[0].legend()

# Distribution des outliers
axes[1].hist(outliers['Amount'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Outliers détectés (n={len(outliers):,})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Montant (€)')
axes[1].set_ylabel('Fréquence')
axes[1].axvline(outliers['Amount'].median(), color='#2c3e50', linestyle='--',
                label=f"Médiane = {outliers['Amount'].median():.1f}€")
axes[1].legend()

plt.suptitle('Comparaison des distributions : Normaux vs Outliers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Étape 5 : Évaluation des performances

Notre IQR détecte des **outliers statistiques** (valeurs extrêmes de Amount).
Le dataset fournit aussi une colonne `Class` qui indique les vraies **fraudes**.

> ⚠️ **Attention** : Un outlier statistique (montant inhabituel) n'est **pas forcément** une fraude, et inversement une fraude peut avoir un montant normal. Les deux concepts sont différents. C'est pourquoi les métriques ci-dessous ne seront pas parfaites — ce qui est normal et attendu.

On va quand même évaluer la correspondance entre nos outliers et les vraies fraudes pour mesurer la pertinence de l'IQR dans ce contexte.

In [ ]:
# ============================================================
# MATRICE DE CONFUSION
# ============================================================
# On compare nos prédictions IQR (outlier ou non) avec les vraies étiquettes
# (fraude ou non) pour construire une matrice de confusion.

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)

# Affichage visuel de la matrice
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=['Normal', 'Outlier'],
            yticklabels=['Normal (vrai)', 'Fraude (vrai)'], ax=ax,
            annot_kws={'size': 14})
ax.set_xlabel('Prédiction IQR', fontsize=12)
ax.set_ylabel('Vérité terrain (Class)', fontsize=12)
ax.set_title('Matrice de confusion : IQR vs Vérité terrain', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nLecture de la matrice :")
print(f"  - Vrais Négatifs  (VN) : {cm[0,0]:,}  (normaux correctement classés normaux)")
print(f"  - Faux Positifs   (FP) : {cm[0,1]:,}  (normaux classés à tort comme outliers)")
print(f"  - Faux Négatifs   (FN) : {cm[1,0]:,}  (fraudes ratées)")
print(f"  - Vrais Positifs  (VP) : {cm[1,1]:,}  (fraudes détectées comme outliers)")

In [ ]:
# ============================================================
# MÉTRIQUES D'ÉVALUATION
# ============================================================
# On calcule les métriques classiques de classification.

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall    = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)

print("="*55)
print("   MÉTRIQUES DE PERFORMANCE (IQR sur Amount)")
print("="*55)
print(f"   Accuracy  (Exactitude)  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"   Precision (Précision)   : {precision:.4f}  ({precision*100:.2f}%)")
print(f"   Recall    (Rappel)      : {recall:.4f}  ({recall*100:.2f}%)")
print(f"   F1-Score                : {f1:.4f}  ({f1*100:.2f}%)")
print("="*55)
print()
print("📝 Interprétation :")
print("   - Precision : Parmi les outliers détectés, combien sont de vraies fraudes ?")
print("   - Recall    : Parmi les vraies fraudes, combien ont été détectées ?")
print("   - F1-Score  : Moyenne harmonique de Precision et Recall.")

In [ ]:
# ============================================================
# RAPPORT DE CLASSIFICATION COMPLET
# ============================================================
print("Rapport de classification détaillé :\n")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraude/Outlier']))

In [ ]:
# ============================================================
# GRAPHIQUE RÉCAPITULATIF DES MÉTRIQUES
# ============================================================
# Diagramme en barres pour visualiser rapidement les métriques.

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metrics_names, metrics_values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)

# Ajout des valeurs sur les barres
for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Métriques de performance — IQR sur Amount', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Étape 6 : Utilisation de la méthode `summary()`

Notre module fournit aussi une méthode `summary()` qui affiche un résumé compact des paramètres appris par le détecteur. C'est pratique pour vérifier rapidement les bornes.

In [ ]:
# ============================================================
# RÉSUMÉ DU DÉTECTEUR (MÉTHODE INTÉGRÉE)
# ============================================================
# Notre classe IQR possède une méthode summary() qui affiche
# automatiquement les paramètres appris.

detector.summary()

## Conclusion

### Ce qu'on a fait dans ce notebook :
1. ✅ **Importé** notre module IQR avec un fallback local (try/except + sys.path)
2. ✅ **Chargé** le dataset Credit Card et exploré la variable `Amount`
3. ✅ **Détecté** les outliers avec `fit()` + `predict()` de notre IQR
4. ✅ **Visualisé** les résultats (scatter, boxplot, histogrammes)
5. ✅ **Évalué** les performances avec matrice de confusion, accuracy, precision, recall, F1

### Limites de l'approche IQR
- L'IQR est une méthode **univariée** : elle analyse chaque variable indépendamment
- Elle détecte des **valeurs extrêmes statistiques**, pas forcément des fraudes
- Pour la détection de fraude, des méthodes **multivariées** (Isolation Forest, etc.) sont plus adaptées

### Points forts de notre implémentation
- API **compatible scikit-learn** (`fit`, `predict`, `fit_predict`)
- Gestion robuste des **NaN** et des **features constantes**
- Méthodes utilitaires : `get_bounds()`, `summary()`